# Amharic Captions retrain — Kaggle (free GPU)

**What you do:** Settings → Accelerator **GPU T4 x1**, Internet **On**, then
**Run All** and leave it. It stops itself after `MAX_HOURS` of training. When
it finishes, download `retrain_output.zip` from the Output pane.

**What it does (recipe of 2026-09-23, TESTING.md 1.2o):**
fine-tunes `badrex/Ethio-ASR-amharic` (the shipped model) on WAXAL Amharic —
top 8 encoder layers + CTC head, fp16, with room-echo and phone-mic
simulation — on transcripts cleaned so numbers are always spelled out (the
2% written as digits taught the model its "5mሰት" habit).

**Honest by construction:** the speakers held out for testing are FIXED in
`tools/retrain/splits/`, so this run can never train on a voice the Mac later
uses as the final exam. The best checkpoint by held-out CER is what's kept,
and the gate below compares it with the shipped model. Nothing here replaces
the shipped model — the Mac re-tests everything first.

## Config — the defaults are the recommended run

| constant | meaning |
|---|---|
| `N_SHARDS`  | WAXAL shards to use (~1,800 clips / ~1 GB audio each; 21 = all) |
| `MAX_HOURS` | training time budget; the session is killed at 12 h total |
| `UNFREEZE`  | top encoder layers to train (+ CTC head) |
| `BATCH` `ACCUM` `LR` `EPOCHS` | optimisation |

In [ ]:
CFG = dict(
    N_SHARDS  = 12,      # ~21k clips; download+prep ~40 min
    MAX_HOURS = 9.0,     # leaves ~2 h for setup, gate and export
    UNFREEZE  = 8,
    BATCH     = 4,
    ACCUM     = 2,       # effective batch 8
    LR        = 3e-5,    # low: we adapt a good model, we don't retrain it
    EPOCHS    = 2,
    MAX_SECS  = 25,      # WAXAL clips are long (median 17.6 s)
    HF_MODEL  = "badrex/Ethio-ASR-amharic",
)
WX    = "/tmp/waxal"                       # big + temporary (not saved)
OUT   = "/kaggle/working/model-retrained"
CT2   = "/kaggle/working/model-ct2-int8-retrain"
ROOT  = "/kaggle/working/amharic-caption"
import os; os.makedirs(WX, exist_ok=True)

In [ ]:
import subprocess, sys
def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=False, text=True, **kw)
    if r.returncode:
        print(f"[fail] rc={r.returncode}: {cmd}")
        raise SystemExit(r.returncode)
    return r

# 1) clone the (public) repo so the existing retrain pipeline runs as-is
if not os.path.isdir(ROOT):
    run(["git", "clone", "--depth", "1",
         "https://github.com/kaleb21-19/amharic_caption", ROOT])
print("repo ready")

# 2) top up packages (torch/torchaudio/pyarrow/pandas ship with the image)
pip = [sys.executable, "-m", "pip", "install", "-q", "--no-input",
       "transformers>=4.52", "soundfile", "ctranslate2"]
for _ in range(3):
    r = subprocess.run(pip, capture_output=True, text=True)
    if r.returncode == 0:
        break
    print(r.stderr[-500:])
else:
    raise SystemExit("pip install failed 3x")

import torch, transformers
print("torch cuda =", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("transformers =", transformers.__version__)

In [ ]:
# 3) FETCH — download N WAXAL Amharic ASR shards (chunked, resumes).
#    This is the same script the repo uses; fast on Kaggle's network.
cmd = ["python3", f"{ROOT}/tools/retrain/01_fetch_waxal.py", "--out", WX,
       "--max-shards", str(CFG["N_SHARDS"])]
run(cmd)
import glob
shards = sorted(glob.glob(f"{WX}/*.parquet"))
tot = sum(os.path.getsize(p) for p in shards)
print(f"[ok] {len(shards)} shards, {tot/1e9:.2f} GB")

In [ ]:
# 4) PREP — parquet -> 16 kHz wavs + manifest.tsv, then split by the FIXED
#    speaker lists in tools/retrain/splits/ (never a per-run carve: the Mac's
#    final exam must use speakers this run never trains on).
run(["python3", f"{ROOT}/tools/retrain/02_prep_waxal.py",
     "--shards", WX, "--wavs", f"{WX}/wavs",
     "--manifest", f"{WX}/manifest.tsv"])
for pq_file in glob.glob(f"{WX}/*.parquet"):   # free the disk for training
    os.remove(pq_file)
run(["python3", f"{ROOT}/tools/retrain/split_by_speakers.py",
     "--manifest", f"{WX}/manifest.tsv", "--out-dir", WX])

In [ ]:
# 5) sanity peek at the training manifest
for line in open(f"{WX}/train.tsv", encoding="utf-8").read().splitlines()[:3]:
    p, _s, t = line.split("\t", 2)
    print(os.path.basename(p), "|", t[:60])

In [ ]:
# 6) FINE-TUNE. Prints a dev CER every 500 steps; the best one is saved to OUT.
#    --dev-manifest also acts as a LEAK GUARD (refuses if any row overlaps).
cmd = ["python3", "-u", f"{ROOT}/tools/retrain/03_finetune_waxal.py",
       "--manifest", f"{WX}/train.tsv",
       "--dev-manifest", f"{WX}/dev.tsv",
       "--src", CFG["HF_MODEL"],
       "--out", OUT,
       "--unfreeze-top-n", str(CFG["UNFREEZE"]),
       "--amp", "--grad-checkpointing",
       "--batch-size", str(CFG["BATCH"]),
       "--grad-accum", str(CFG["ACCUM"]),
       "--lr", str(CFG["LR"]),
       "--epochs", str(CFG["EPOCHS"]),
       "--max-secs", str(CFG["MAX_SECS"]),
       "--mask-time-prob", "0.05",
       "--reverb-prob", "0.3", "--narrowband-prob", "0.3",
       "--eval-every", "500", "--eval-rows", "80",
       "--max-hours", str(CFG["MAX_HOURS"])]
run(cmd)

In [ ]:
# 7) GATE — shipped model vs retrained, on the held-out speakers.
#    exit 0 = KEEP (retrained <= shipped), 1 = REJECT.
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
r = subprocess.run(["python3", f"{ROOT}/tools/retrain/04_eval_wer.py",
                    "--manifest", f"{WX}/dev.tsv",
                    "--current", CFG["HF_MODEL"],
                    "--candidate", OUT, "--device", dev],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-2000:] if r.returncode else "")
VERDICT = "KEEP" if r.returncode == 0 else "REJECT"
print("\n>>> GATE VERDICT:", VERDICT)

## Reading the gate

* **KEEP** — better than the shipped model on held-out speakers → the export
  cell builds the product model. Download `retrain_output.zip`.
* **REJECT** — no better. Nothing changes for customers. Download the zip
  anyway (it has the training log numbers) and share it; that is still a
  useful result.

Either way the Mac re-tests on the real clips, numbers, phone and echo
conditions before anything ships.

In [ ]:
# 8) EXPORT — CTranslate2 int8, the exact layout the product build packages.
import shutil, glob
assert os.path.isdir(OUT), "train first"
env = dict(os.environ, MODEL_SRC=OUT, MODEL_DST=CT2)
run(["bash", f"{ROOT}/tools/make_model_ct2_int8.sh"], env=env)

# bundle the HF checkpoint + ct2 int8 + gate summary for download
summary = f"\n".join([l for l in r.stdout.splitlines() if "[result]" in l])
open("/kaggle/working/SUMMARY.txt", "w", encoding="utf-8").write(
    f"gate verdict: {VERDICT}\n{summary}\n")
for what, src in (["model-retrained", OUT], ["model-ct2-int8-retrain", CT2],
                  ["SUMMARY.txt", "/kaggle/working/SUMMARY.txt"]):
    dst = f"/kaggle/working/bundle/{what}"
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
shutil.make_archive("/kaggle/working/retrain_output", "zip",
                    "/kaggle/working/bundle")
import os
print("[ok] /kaggle/working/retrain_output.zip",
      f"{os.path.getsize('/kaggle/working/retrain_output.zip')/1e6:.0f} MB")

## Installing the result back on the Mac

If (and only if) the gate said **KEEP**: download `retrain_output.zip` from the
notebook Output pane, unzip it, and on the repo — **one command**:

```bash
bash tools/retrain/install_retrained.sh /path/to/unzipped/retrain_output
```

It refuses a REJECT bundle, backs up `tools/stage/model-ct2-int8` to
`model-ct2-int8.prev`, swaps in the retrained int8 model, re-scores old vs new
on `tools/stage/waxal/holdout.tsv` (the same `04_eval_wer.py` as the gate) and
auto-reverts on regression, then rebuilds mac-arm64 + mac-x64 + win-x64
(~25-30 min). Rollback: `mv tools/stage/model-ct2-int8.prev tools/stage/model-ct2-int8`.

Then re-run the honest fixture set (same 40 real clips that gave 0.227). Keep
the swap only if the honest WER is at least as good as 0.227 — a WAXAL-dev win
is encouraging but the real measurements are what ship.

The model outputs are gitignored; nothing here needs to be committed. This
exercise costs nothing (free Kaggle GPU session).